# Vulnerability 3 — Path Traversal in Model Loading

This notebook demonstrates **path traversal** in an ML API that loads models based on user input.

This is common in multi‑model serving APIs where clients specify which model to use.


## 1. Vulnerable API Code

We simulate a Flask API that loads a model based on a `model_name` field in the request JSON.


In [1]:
print("="*70)
print("VULNERABILITY 3: PATH TRAVERSAL IN MODEL LOADING")
print("="*70)

vulnerable_api_code = '''
from flask import Flask, request
import pickle

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    # VULNERABILITY: User input used directly in file path
    model_name = request.json.get('model_name')
    model_path = f"models/{model_name}.pkl"

    # No path validation!
    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    predictions = model.predict(request.json.get('data'))
    return {'predictions': predictions.tolist()}
'''

print("\n[Vulnerable Code Pattern]\n")
print(vulnerable_api_code)

VULNERABILITY 3: PATH TRAVERSAL IN MODEL LOADING

[Vulnerable Code Pattern]


from flask import Flask, request
import pickle

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    # VULNERABILITY: User input used directly in file path
    model_name = request.json.get('model_name')
    model_path = f"models/{model_name}.pkl"

    # No path validation!
    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    predictions = model.predict(request.json.get('data'))
    return {'predictions': predictions.tolist()}



## 2. Attack Example

An attacker can send a request like:
```json
{
  "model_name": "../../../../etc/passwd",
  "data": [...]
}
```

Resulting path:
```text
models/../../../../etc/passwd.pkl
```

This can be used to:
- Read arbitrary files
- Load malicious models from unexpected locations
- Bypass access controls


In [2]:
print("\n[Attack Example]")
print('Attacker sends request with:')
print('  {"model_name": "../../../../etc/passwd"}')
print("Resulting path: models/../../../../etc/passwd.pkl")

print("\n⚠️  VULNERABILITY: Path Traversal")
print("Impact:")
print("  • Read arbitrary files from filesystem")
print("  • Load malicious models from unexpected locations")
print("  • Bypass access controls and authentication")
print("  • Common in ML APIs that serve multiple models")


[Attack Example]
Attacker sends request with:
  {"model_name": "../../../../etc/passwd"}
Resulting path: models/../../../../etc/passwd.pkl

⚠️  VULNERABILITY: Path Traversal
Impact:
  • Read arbitrary files from filesystem
  • Load malicious models from unexpected locations
  • Bypass access controls and authentication
  • Common in ML APIs that serve multiple models


## 3. Detection & Prevention

### 🔍 Detection
- Semgrep or other static analysis tools can detect flows:
  - **User input → file path → `open()` / `pickle.load()`**

### ✅ Prevention
- Use a **whitelist** of allowed model names
- Map logical model IDs to safe file paths
- Reject any input containing `..`, `/`, `\\`, or absolute paths
- Avoid exposing raw filesystem structure to clients
